### **Chapter 9.7: Inside the DeePC Solution - Anatomy of $g$, Equivalence, Data, and Computation**

Chapters 9.1-9.3 established that DeePC works on the flat Mountain Car and coincides with model-based MPC. This notebook opens the box on the **noise-free linear case** and asks four questions that are usually skipped:

1. **What is $g$?** Which pieces of the recorded trajectory does the controller glue together to predict the future? (*Willems' fundamental lemma, made visible*)
2. **When exactly is DeePC equivalent to MPC** - and how does the equivalence break when the data are too short, the history is too short, or the plant is not LTI?
3. **How much data is needed?** The Hankel matrix has full row rank only if $T \ge (m+1)(T_{\mathrm{ini}}+N+n)-1$.
4. **What does it cost online?** The DeePC QP has as many decision variables as Hankel *columns*; MPC has $mN$. How do solve times scale with $T$ and $T_{\mathrm{ini}}$?

As in the whole chapter the measured output is the position only, $y = p$. The model-based reference is an MPC in ARX form (`ARXPredictiveController`) that uses the *exact* zero-order-hold model $p_{k+1} = 2p_k - p_{k-1} + \tfrac{\Delta t^2}{2}(u_k + u_{k-1})$ and the **same OSQP backend, cost and constraints** as DeePC, so that every difference we measure is due to the trajectory representation and nothing else.

In [ ]:
import sys
import os
import io
import time
import contextlib
import numpy as np
import matplotlib.pyplot as plt

sys.path.append(os.path.abspath(".."))
from utils.env import *
from ex9_DeePC.deepc_utils import *

# ---- figure style for the tutorial paper -------------------------------------
OKABE_ITO = ["#0072B2", "#E69F00", "#009E73", "#D55E00", "#CC79A7", "#56B4E9", "#F0E442", "#000000"]
plt.rcParams.update({
    "font.size": 11, "axes.labelsize": 11, "axes.titlesize": 11, "legend.fontsize": 9,
    "xtick.labelsize": 10, "ytick.labelsize": 10, "figure.dpi": 110, "savefig.dpi": 300,
    "axes.grid": True, "grid.alpha": 0.3, "lines.linewidth": 1.8,
    "axes.prop_cycle": plt.cycler(color=OKABE_ITO),
})
FIG_DIR = "figures"
os.makedirs(FIG_DIR, exist_ok=True)

def savefig(fig, name):
    fig.savefig(os.path.join(FIG_DIR, f"{name}.pdf"), bbox_inches="tight")
    fig.savefig(os.path.join(FIG_DIR, f"{name}.png"), bbox_inches="tight")

In [ ]:
freq = 20
dt = 1.0 / freq
N = 20
T_ini = 4
n_states = 2          # system order n
m_inputs = 1          # number of inputs m
L = T_ini + N         # Hankel depth used by DeePC
t_terminal = 6.0

initial_state = np.array([-0.5, 0.0])
target_state = np.array([0.6, 0.0])

env = Env(1, initial_state, target_state, input_lbs=-1.0, input_ubs=1.0)
dynamics = Dynamics(env)

Q_y = np.array([[1.0]])
R = np.array([[0.1]])
Qf_y = Q_y

a_true, b_true = true_arx_double_integrator(dt)
print(f"exact ARX model: p_k+1 = {a_true[0]:.0f} p_k {a_true[1]:+.0f} p_k-1 + {b_true[0]:.5f} u_k + {b_true[1]:.5f} u_k-1")

def make_oracle():
    return ARXPredictiveController(env, dynamics, a_true, b_true, Q_y, R, Qf_y, freq, N, name="ARX-MPC (true model)")

def make_deepc(u_data, y_data, T_ini=T_ini, lambda_g=1e-10, lambda_y=None, N=N, enforce_current_output=True):
    return DeePCController(env, dynamics, u_data, y_data, Q_y, R, Qf_y, freq, N, T_ini=T_ini,
                           lambda_g=lambda_g, lambda_y=lambda_y, output_indices=[0],
                           enforce_current_output=enforce_current_output,
                           history_initialization='equilibrium', name='DeePC', verbose=False)

def run_closed_loop(controller, plant=dynamics, env_run=env, t_end=t_terminal):
    # Standing still with u = 0 is an exact trajectory of the flat plant, so the
    # default 'equilibrium' history initialization is consistent here.
    x = env_run.init_state.copy()
    states, inputs, solve_times, failures = [x.copy()], [], [], 0
    for k in range(int(freq * t_end)):
        try:
            with contextlib.redirect_stdout(io.StringIO()):
                u = controller.compute_action(x, k)
            u = u[0] if isinstance(u, tuple) else u
            solve_times.append(controller.last_solve_time)
        except RuntimeError:
            failures += 1
            u = np.zeros(1)
            if isinstance(controller, DeePCController):
                controller.initialize_history(x, mode='equilibrium')
        u = np.clip(np.asarray(u, dtype=float).reshape(-1), env_run.input_lbs, env_run.input_ubs)
        x = plant.one_step_forward(x, u, dt)
        states.append(x.copy()); inputs.append(u.copy())
    return np.asarray(states), np.asarray(inputs), np.asarray(solve_times), failures

def closed_loop_cost(states, inputs):
    dp = states[:, 0] - target_state[0]
    du = inputs[:, 0] - float(dynamics.get_equilibrium_input(target_state))
    return float(np.sum(Q_y[0, 0] * dp[:-1]**2 + R[0, 0] * du**2) + Qf_y[0, 0] * dp[-1]**2)

T_data = 400
u_data, y_data = collect_deepc_data(env, dynamics, freq=freq, n_samples=T_data, excitation_amplitude=0.8,
                                    initial_state=target_state, seed=1, output_indices=[0])
print(f"offline data: T = {T_data} inputs, Hankel depth L = T_ini + N = {L}, columns = {T_data - L + 1}")

### **Part 1: Anatomy of the Behavioral Coefficient $g$**

DeePC predicts with $\begin{bmatrix}u_{\mathrm{ini}}\\y_{\mathrm{ini}}\\u_f\\y_f\end{bmatrix} = \begin{bmatrix}U_p\\Y_p\\U_f\\Y_f\end{bmatrix} g$. Column $j$ of the Hankel matrix is the length-$L$ window of the recorded trajectory that starts at sample $j$. So $g_j$ is literally the weight with which *the piece of the experiment recorded between $t_j$ and $t_j + L\Delta t$* enters the current prediction. Willems' lemma says that, for persistently exciting data, every length-$L$ trajectory of the system is such a linear combination.

We solve one DeePC problem from the initial state and look at the $g$ that OSQP returns.

In [ ]:
def solve_once(controller, x=initial_state):
    with contextlib.redirect_stdout(io.StringIO()):
        u0, y_pred, u_pred = controller.compute_action(x, 0)
    return u0, y_pred[:, 0], u_pred[:, 0], controller.last_g.copy()

ctrl = make_deepc(u_data, y_data)
u0, y_pred, u_pred, g = solve_once(ctrl)

# The same future input applied to the exact model gives the "true" prediction.
oracle = make_oracle()
oracle.initialize_history(initial_state)
y_true = np.concatenate([[initial_state[0]], oracle.Phi @ np.concatenate([oracle.y_history[::-1], oracle.u_history[::-1]])
                          + oracle.Gamma @ (u_pred - oracle.equilibrium_input)]) + 0.0
y_true[1:] += oracle.target_output

H_stack = np.vstack([ctrl.Up, ctrl.Yp, ctrl.Uf, ctrl.Yf])
print(f"stacked Hankel matrix: {H_stack.shape}, rank = {np.linalg.matrix_rank(H_stack)} = m*L + n = {m_inputs*L + n_states}")
print(f"dimension of the affine set of valid g: {H_stack.shape[1] - np.linalg.matrix_rank(H_stack)}")
print(f"||g||_2 = {np.linalg.norm(g):.3f}, number of |g_j| > 1% of max: {int(np.sum(np.abs(g) > 0.01 * np.abs(g).max()))} of {g.size}")
print(f"max |y_pred - y_true model| = {np.abs(y_pred - y_true).max():.2e} m")

In [ ]:
t_data = np.arange(T_data + 1) * dt
t_cols = np.arange(g.size) * dt                 # start time of the window represented by column j
top = np.argsort(-np.abs(g))[:5]                # five most heavily weighted windows

fig, ax = plt.subplots(3, 1, figsize=(9, 7.5), height_ratios=[1.1, 1, 1])
markerline, stemlines, baseline = ax[0].stem(t_cols, g, basefmt=" ")
plt.setp(stemlines, linewidth=0.8, color=OKABE_ITO[0]); plt.setp(markerline, markersize=2.5, color=OKABE_ITO[0])
ax[0].scatter(t_cols[top], g[top], s=45, facecolors="none", edgecolors=OKABE_ITO[3], linewidths=1.5, zorder=3, label="5 largest $|g_j|$")
ax[0].set_ylabel(r"$g_j$"); ax[0].set_title(r"Behavioral coefficient $g$ over the start time $t_j$ of the data window it weights"); ax[0].legend(loc="upper right")

ax[1].plot(t_data[:-1], u_data[:, 0], color="0.4", lw=1.0, label=r"recorded input $u^{d}$")
ax[1].set_ylabel("input"); ax[1].legend(loc="upper right")
ax[2].plot(t_data, y_data[:, 0], color="0.4", lw=1.0, label=r"recorded output $y^{d}=p^{d}$")
ax[2].set_ylabel("position"); ax[2].set_xlabel("time in the offline experiment (s)"); ax[2].legend(loc="upper right")
for j in top:
    for a_ in ax[1:]:
        a_.axvspan(t_cols[j], t_cols[j] + L * dt, color=OKABE_ITO[3], alpha=0.12)
ax[1].set_title("The five most heavily weighted windows of the experiment (shaded, length $L\\,\\Delta t$)")
plt.tight_layout()
savefig(fig, "9.7_g_anatomy")
plt.show()

In [ ]:
t_f = np.arange(N + 1) * dt
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
ax[0].plot(t_f, y_true, "k", lw=3, alpha=0.35, label="exact model, same $u_f$")
ax[0].plot(t_f, y_pred, "--", label=r"DeePC: $Y_f\,g$")
ax[0].axhline(target_state[0], color=OKABE_ITO[1], ls=":", label="target")
ax[0].set_xlabel("prediction time (s)"); ax[0].set_ylabel("position"); ax[0].legend(); ax[0].set_title("Predicted output")
ax[1].step(t_f[:-1], u_pred, where="post", label=r"DeePC: $U_f\,g$")
ax[1].axhline(env.input_ubs, color="0.5", ls="--", lw=1); ax[1].axhline(env.input_lbs, color="0.5", ls="--", lw=1)
ax[1].set_xlabel("prediction time (s)"); ax[1].set_ylabel("input"); ax[1].legend(); ax[1].set_title("Predicted input")
plt.tight_layout()
savefig(fig, "9.7_g_prediction")
plt.show()

Three things to read off:

- $g$ is **dense**: with $\lambda_g \to 0$ the QP returns the minimum-norm element of a $\bigl(\#\text{columns} - (mL+n)\bigr)$-dimensional affine set of equally valid coefficients. The prediction is a weighted average of *many* overlapping windows, not a lookup of one similar episode. Any other element of that set would produce exactly the same $u_f, y_f$ - that is the content of the fundamental lemma.
- The heavily weighted windows are the ones whose recorded input pattern resembles what the controller wants to do now (a saturated push followed by a reversal); their *outputs* then tell it what will happen.
- The composed prediction agrees with the exact model to solver precision: the Hankel matrix *is* the model.

#### **Example 1.1: What $\lambda_g$ does on clean data**

Chapter 9.5 uses $\lambda_g$ to fight noise. Two regimes must be distinguished. With **hard** history constraints, $\lambda_g$ only selects *which* element of the affine set of valid $g$ is returned; every element produces the same trajectory, so the prediction cannot change - that is the fundamental lemma at work. With the **soft** history fit ($\lambda_y < \infty$) the true trajectory is no longer forced, and $\lambda_g\|g\|^2$ pulls the prediction towards the data even when the data are perfect: a bias that grows with $\lambda_g$.

In [ ]:
def prediction_bias(controller):
    # max deviation of the DeePC output prediction from the exact model driven by the same u_f
    u0_l, y_pred_l, u_pred_l, g_l = solve_once(controller)
    o = make_oracle(); o.initialize_history(initial_state)
    y_model = o.Phi @ np.concatenate([o.y_history[::-1], o.u_history[::-1]]) + o.Gamma @ (u_pred_l - o.equilibrium_input) + o.target_output
    return np.linalg.norm(g_l), float(np.abs(y_pred_l[1:] - y_model).max()), float(u0_l[0])

lambda_values = [1e-10, 1e-4, 1e-2, 1.0, 1e2]
print(f"{'lambda_g':>10} | {'hard history: ||g||':>19} {'pred. bias (m)':>15} {'u_0':>7} | {'soft history (lambda_y=1e4): ||g||':>34} {'pred. bias (m)':>15} {'u_0':>7}")
for lam in lambda_values:
    gh, bh, uh = prediction_bias(make_deepc(u_data, y_data, lambda_g=lam))
    gs, bs, us = prediction_bias(make_deepc(u_data, y_data, lambda_g=lam, lambda_y=1e4))
    print(f"{lam:10.0e} | {gh:19.3f} {bh:15.2e} {uh:7.3f} | {gs:34.3f} {bs:15.2e} {us:7.3f}")

With hard history constraints the prediction bias stays at machine precision for every $\lambda_g$ while $\|g\|$ shrinks from 0.98 to 0.10: the regularizer only chooses a representative of the same trajectory. Only at $\lambda_g = 10^2$ does the *control* change ($u_0 = 0.38$ instead of the saturated $1.0$), because $\lambda_g\|g\|^2$ then dominates the tracking cost - the prediction is still exact, but the optimizer prefers "small" trajectories. With the soft history fit the prediction is biased by about 5 cm at the very first step, essentially independently of $\lambda_g$: the optimizer buys a lower tracking cost by misfitting the (perfect) history slightly, a trade-off set by $\lambda_y$, not by $\lambda_g$. On noise-free data the hard formulation is therefore the right one; the soft one is a tool for Chapter 9.8.

### **Part 2: When Is DeePC Equivalent to MPC - and When Not?**

For a controllable LTI system with persistently exciting data of sufficient length, the DeePC QP and the model-based QP have the same feasible set and the same optimal value (Coulson et al., Thm. 5.1). We compare closed loops against the exact-model ARX-MPC in four situations:

| case | what changes | expectation |
|---|---|---|
| A. nominal | $T = 400$, $T_{\mathrm{ini}} = 4$, flat plant | identical inputs |
| B. minimal data | $T$ exactly at the bound $(m+1)(T_{\mathrm{ini}}+N+n)-1$ (below it the Hankel matrix is rank deficient and the controller refuses the data) | still identical inputs |
| C. short history | $T_{\mathrm{ini}} = 1 <$ lag $= 2$ (textbook output-feedback form of Chapter 9.4, current output not appended) | the hidden velocity is not pinned down, prediction is not unique |
| D. nonlinear plant | data collected *on the bumpy plant* $h = k\cos(18p)$, hard DeePC ($\lambda_g \to 0$) vs. MPC with the flat model | difference grows with $k$ |

For D the data are collected in closed loop around the target (Chapter 9.6) so that they stay in the region of interest.

In [ ]:
T_min = (m_inputs + 1) * (T_ini + N + n_states) - 1
print(f"data-length bound for full row rank: T >= (m+1)(T_ini+N+n)-1 = {T_min}")

def compare(deepc_ctrl, plant=dynamics, env_run=env, t_end=4.0):
    X_d, U_d, _, fails = run_closed_loop(deepc_ctrl, plant, env_run, t_end)
    X_m, U_m, _, _ = run_closed_loop(make_oracle(), plant, env_run, t_end)
    return X_d, U_d, X_m, U_m, fails, float(np.abs(U_d - U_m).max())

cases = {}
# A. nominal
cases["A: nominal\n$T$=400, $T_{ini}$=4"] = compare(make_deepc(u_data, y_data))
# B. data length exactly at the bound; below it the controller refuses to build the Hankel matrix
try:
    make_deepc(u_data[:T_min - 6], y_data[:T_min - 6 + 1])
except ValueError as e:
    print(f"T = {T_min - 6} < {T_min}: {e}")
cases[f"B: minimal data\n$T$={T_min}"] = compare(make_deepc(u_data[:T_min], y_data[:T_min + 1]))
# C. short history
# C. short history: one past sample only. (With enforce_current_output=True the current position
#    would be appended to the window and two positions already determine the velocity - see 9.4.)
cases["C: short history\n$T_{ini}$=1 < lag 2"] = compare(make_deepc(u_data, y_data, T_ini=1, enforce_current_output=False))
# D. nonlinear plant with data from that plant
import scipy.linalg
A_d, B_d = dynamics.get_linearized_AB_discrete(target_state, np.zeros(1), dt)
P_lqr = scipy.linalg.solve_discrete_are(A_d, B_d, np.diag([1.0, 0.0]), R)
K_lqr = np.linalg.solve(R + B_d.T @ P_lqr @ B_d, B_d.T @ P_lqr @ A_d)
for k_bump in [0.001, 0.003]:
    env_b = Env(3, initial_state, target_state, param=k_bump, input_lbs=-1.0, input_ubs=1.0)
    dyn_b = Dynamics(env_b)
    u_b, y_b = collect_deepc_data_closed_loop(env_b, dyn_b, freq=freq, n_samples=T_data, feedback_gain=K_lqr,
                                              excitation_amplitude=0.3, seed=1, output_indices=[0])
    c = DeePCController(env_b, dyn_b, u_b, y_b, Q_y, R, Qf_y, freq, N, T_ini=T_ini, lambda_g=1e-10,
                        output_indices=[0], history_initialization='equilibrium', verbose=False)
    cases[f"D: bumpy plant\n$k$={k_bump}"] = compare(c, dyn_b, env_b)

print(f"\n{'case':<34} {'max |u_DeePC - u_MPC|':>22} {'QP failures':>12}")
for name, (X_d, U_d, X_m, U_m, fails, dev) in cases.items():
    print(f"{name.replace(chr(10), ' '):<34} {dev:22.3e} {fails:12d}")

In [ ]:
names = list(cases.keys())
devs = [cases[n][5] for n in names]
fig, ax = plt.subplots(1, 2, figsize=(12.5, 3.8), width_ratios=[1.35, 1])
bars = ax[0].bar(range(len(names)), np.maximum(devs, 1e-12), color=[OKABE_ITO[2]] + [OKABE_ITO[3]] * (len(names) - 1))
ax[0].set_yscale("log"); ax[0].set_xticks(range(len(names))); ax[0].set_xticklabels([n.replace(": ", ":\n") for n in names], fontsize=8)
ax[0].set_ylabel(r"$\max_k \; |u^{\mathrm{DeePC}}_k - u^{\mathrm{MPC}}_k|$")
ax[0].axhline(1e-5, color="0.5", ls=":", lw=1); ax[0].text(len(names) - 0.5, 1.4e-5, "solver tolerance", ha="right", fontsize=8, color="0.4")
ax[0].set_title("Input deviation from the exact-model MPC")

t_x = np.arange(int(freq * 4.0) + 1) * dt
for i, n_ in enumerate([names[0], names[2]]):  # nominal and short-history cases
    X_d, U_d, X_m, U_m, _, _ = cases[n_]
    ax[1].plot(t_x, X_m[:, 0], color="k", lw=3, alpha=0.3, label="MPC (exact model)" if i == 0 else None)
    ax[1].plot(t_x, X_d[:, 0], "--", color=OKABE_ITO[i * 3], label="DeePC, " + n_.split("\n")[1])
ax[1].axhline(target_state[0], color=OKABE_ITO[1], ls=":")
ax[1].set_xlabel("time (s)"); ax[1].set_ylabel("position"); ax[1].legend(); ax[1].set_title("Closed loop: nominal vs. too short history")
plt.tight_layout()
savefig(fig, "9.7_equivalence_breakdown")
plt.show()

- **A** reproduces the MPC to solver tolerance.
- **B**: the bound is sharp in both directions. With $T < (m+1)(L+n)-1$ the input Hankel matrix has fewer columns than rows and cannot be persistently exciting (the controller rejects the data); with $T$ exactly at the bound the loop is already identical to the MPC. More data adds nothing to the noise-free solution - it only enlarges the QP (Part 3).
- **C**: with $T_{\mathrm{ini}}=1$ the constraints leave the hidden velocity free (Chapter 9.4). The QP then selects the *cheapest* consistent velocity - an optimistic prediction - and the closed loop overshoots. No amount of data fixes this; it is a property of the history length, not of $T$.
- **D**: on the bumpy plant there is no LTI behavior to span. Both controllers are "wrong", but differently: MPC uses the flat model, DeePC the (full-rank) nonlinear Hankel matrix with an almost unregularized $g$. The deviation grows with $k$ and is already of the order of the input bound for a 3 mm bump - Chapter 9.6 shows why regularization is then indispensable.

### **Part 3: Data Requirement and Online Computation**

**Data.** A necessary condition for $\begin{bmatrix}U_p\\U_f\end{bmatrix}$ (depth $L+n$ for the PE check) to have full row rank is at least as many columns as rows:

$$
T - (L+n) + 1 \;\ge\; m(L+n) \quad\Longleftrightarrow\quad T \;\ge\; (m+1)(T_{\mathrm{ini}}+N+n) - 1 .
$$

**Computation.** The DeePC decision variable is $g \in \mathbb{R}^{T-L+1}$: it *grows with the data*, and the Hessian $Y_f^\top \bar Q Y_f + U_f^\top \bar R U_f + \lambda_g I$ is dense. The model-based QP has $mN$ variables regardless of how much data were used to identify the model. We sweep $T$ from below the bound to $T = 2000$ and record, for the same closed-loop task, the Hankel rank, the cost relative to the exact-model MPC, and the OSQP solve time per step (`solve_time`, excluding Python overhead; both controllers use identical OSQP settings with warm start and polishing).

In [ ]:
T_values = np.unique(np.concatenate([np.arange(T_min - 8, T_min + 9, 2), [60, 80, 100, 150, 200, 300, 400, 600, 800, 1200, 1600, 2000]])).astype(int)
u_long, y_long = collect_deepc_data(env, dynamics, freq=freq, n_samples=int(T_values.max()), excitation_amplitude=0.8,
                                    initial_state=target_state, seed=1, output_indices=[0])

X_m, U_m, t_m, _ = run_closed_loop(make_oracle())
J_mpc = closed_loop_cost(X_m, U_m)
mpc_solve_us = np.median(t_m) * 1e6

results_T = []
for T in T_values:
    u_T, y_T = u_long[:T], y_long[:T + 1]
    rank, rows_pe = persistent_excitation_rank(u_T, L + n_states)
    t0 = time.perf_counter()
    try:
        c = make_deepc(u_T, y_T)
    except ValueError as e:           # PE check fails inside setup()
        results_T.append((T, rank, rows_pe, np.nan, np.nan, np.nan, np.nan, int(freq * t_terminal)))
        continue
    setup_ms = (time.perf_counter() - t0) * 1e3
    X_d, U_d, t_d, fails = run_closed_loop(c)
    results_T.append((T, rank, rows_pe, closed_loop_cost(X_d, U_d), np.median(t_d) * 1e6, T - L + 1, setup_ms, fails))

print(f"exact-model MPC: cost {J_mpc:.4f}, median OSQP solve time {mpc_solve_us:.1f} us, decision variables {N}")
print(f"\n{'T':>5} {'rank(H_L+n(u))':>15} {'columns':>8} {'cost - cost_MPC':>16} {'solve (us)':>11} {'setup (ms)':>11} {'fails':>6}")
for T, rank, rows_pe, J, t_us, ncol, setup_ms, fails in results_T:
    Jd = "   infeasible" if np.isnan(J) else f"{J - J_mpc:16.2e}"
    print(f"{T:5d} {rank:8d}/{rows_pe:<6d} {ncol if not np.isnan(J) else T-L+1:8d} {Jd:>16} {t_us:11.1f} {setup_ms:11.2f} {fails:6d}")

In [ ]:
Ts = np.array([r[0] for r in results_T]); ranks = np.array([r[1] for r in results_T]); rows_pe = results_T[0][2]
Js = np.array([r[3] for r in results_T]); t_us = np.array([r[4] for r in results_T]); ncols = Ts - L + 1

fig, ax = plt.subplots(1, 3, figsize=(13, 3.6))
ax[0].plot(Ts, ranks, marker="o", ms=3)
ax[0].axhline(rows_pe, color="0.5", ls="--", lw=1, label=f"full row rank = {rows_pe}")
ax[0].axvline(T_min, color=OKABE_ITO[3], ls=":", label=f"bound $T$ = {T_min}")
ax[0].set_xscale("log"); ax[0].set_xlabel("data length $T$"); ax[0].set_ylabel(r"rank $H_{L+n}(u^d)$"); ax[0].legend(); ax[0].set_title("Persistency of excitation")

ok = ~np.isnan(Js)
ax[1].semilogy(Ts[ok], np.maximum(np.abs(Js[ok] - J_mpc), 1e-12), marker="o", ms=3, label="|cost - cost$_{MPC}$|")
ax[1].scatter(Ts[~ok], np.full((~ok).sum(), 1e1), marker="x", color=OKABE_ITO[3], label="infeasible")
ax[1].axvline(T_min, color=OKABE_ITO[3], ls=":")
ax[1].set_xscale("log"); ax[1].set_xlabel("data length $T$"); ax[1].set_ylabel("closed-loop cost deviation"); ax[1].legend(); ax[1].set_title("Equivalence vs. data length")

ax[2].loglog(ncols[ok], t_us[ok], marker="o", ms=3, label=f"DeePC ($g \\in \\mathbb{{R}}^{{T-L+1}}$)")
ref = ncols[ok].astype(float)
ax[2].loglog(ref, t_us[ok][5] * (ref / ref[5])**2, color="0.6", ls="--", lw=1, label=r"$\propto (T-L+1)^2$")
ax[2].loglog(ref, t_us[ok][5] * (ref / ref[5])**3, color="0.6", ls="-.", lw=1, label=r"$\propto (T-L+1)^3$")
ax[2].axhline(mpc_solve_us, color=OKABE_ITO[1], ls="-", label=f"MPC, {N} variables")
ax[2].set_xlabel("Hankel columns $T-L+1$"); ax[2].set_ylabel("OSQP solve time per step ($\\mu$s)"); ax[2].legend(); ax[2].set_title("Online cost")
plt.tight_layout()
savefig(fig, "9.7_data_and_compute_vs_T")
plt.show()

The bound is sharp: below $T = 51$ the input Hankel matrix is rank deficient and the controller rejects the data; from $T = 51$ on the closed loop coincides with the exact-model MPC (cost deviation $\sim 10^{-7}$) and *stays* there - more data change nothing but the QP size. The OSQP solve time grows from about 90 $\mu$s at the bound to 1.7 ms at $T=400$ and 60 ms at $T=2000$, i.e. between quadratically and cubically in the number of columns (dense Hessian, dense KKT factorization), while the model-based QP with 20 variables needs 13 $\mu$s regardless of $T$. The one-off setup (Hankel matrices + OSQP factorization) grows from 1 ms to 0.9 s.

#### **Example 3.1: History length $T_{\mathrm{ini}}$**

Longer histories do not change the solution on noise-free data (any $T_{\mathrm{ini}} \ge$ lag pins down the state), but they change the QP: more equality rows, deeper Hankel matrix, fewer columns for the same $T$. Starting at the smallest valid value we double $T_{\mathrm{ini}}$ and record cost and solve time; for each $T_{\mathrm{ini}}$ we also report the data-length bound that it implies.

In [ ]:
T_ini_values = [2, 4, 8, 16, 32, 64]
u_fix, y_fix = u_long[:400], y_long[:401]
print(f"{'T_ini':>6} {'bound T_min':>12} {'columns':>8} {'eq. rows':>9} {'cost - cost_MPC':>16} {'solve (us)':>11}")
results_Tini = []
for Ti in T_ini_values:
    c = make_deepc(u_fix, y_fix, T_ini=Ti)
    X_d, U_d, t_d, fails = run_closed_loop(c)
    bound = (m_inputs + 1) * (Ti + N + n_states) - 1
    results_Tini.append((Ti, bound, c.Up.shape[1], c._n_eq, closed_loop_cost(X_d, U_d) - J_mpc, np.median(t_d) * 1e6))
    print(f"{Ti:6d} {bound:12d} {c.Up.shape[1]:8d} {c._n_eq:9d} {results_Tini[-1][4]:16.2e} {results_Tini[-1][5]:11.1f}")

fig, ax = plt.subplots(1, 2, figsize=(9, 3.4))
ax[0].semilogy(T_ini_values, np.maximum([abs(r[4]) for r in results_Tini], 1e-12), marker="o")
ax[0].set_xscale("log", base=2); ax[0].set_xlabel(r"$T_{\mathrm{ini}}$"); ax[0].set_ylabel("|cost - cost$_{MPC}$|"); ax[0].set_title("Solution unchanged for $T_{ini} \\geq$ lag")
ax[1].plot(T_ini_values, [r[5] for r in results_Tini], marker="o", label="DeePC")
ax[1].axhline(mpc_solve_us, color=OKABE_ITO[1], label="MPC")
ax[1].set_xscale("log", base=2); ax[1].set_xlabel(r"$T_{\mathrm{ini}}$"); ax[1].set_ylabel("OSQP solve time ($\\mu$s)"); ax[1].legend(); ax[1].set_title("Solve time at fixed $T$ = 400")
plt.tight_layout()
savefig(fig, "9.7_compute_vs_Tini")
plt.show()

As expected, the closed loop is identical for every $T_{\mathrm{ini}} \ge 2$. The solve time grows only mildly (1.7 ms to 2.1 ms from $T_{\mathrm{ini}}=2$ to $64$): the number of columns shrinks slightly, the number of equality rows grows, and the dense Hessian dimension is unchanged. $T_{\mathrm{ini}}$ is therefore cheap on clean data; its real role appears with noise and nonlinearity (Chapters 9.6 and 9.8), where a longer window averages disturbances.

<blockquote style="padding: 18px 20px; margin: 1.2em 0; background: rgba(56, 139, 253, 0.12); border-left: 4px solid rgba(56, 139, 253, 0.85); border-radius: 6px; color: inherit !important;">

##### **Takeaway: DeePC pays for model-freeness with a data-sized QP**

On a noise-free LTI plant with enough persistently exciting data, DeePC is *exactly* the model-based MPC: $g$ is a dense, non-unique blend of many recorded windows, and the composed prediction agrees with the model to solver precision. The equivalence needs $T \ge (m+1)(T_{\mathrm{ini}}+N+n)-1$ and $T_{\mathrm{ini}} \ge$ lag, and it is lost on a nonlinear plant. The price is online computation: the QP dimension equals the number of Hankel columns and the solve time grows between quadratically and cubically with it (90 $\mu$s at the bound, 60 ms at $T = 2000$), whereas MPC's $mN$ variables do not depend on the data at all (13 $\mu$s here). In practice this is why DeePC implementations either keep $T$ close to the bound, or compress the data (SVD/low-rank Hankel, "DeePC in the shallows") before going online.
</blockquote>

**References:** Coulson, Lygeros, and Dörfler, *Data-Enabled Predictive Control: In the Shallows of the DeePC*, ECC 2019 (Thm. 5.1); Willems, Rapisarda, Markovsky, and De Moor, *A note on persistency of excitation*, Systems & Control Letters 2005; Markovsky and Dörfler, *Behavioral systems theory in data-driven analysis, signal processing, and control*, Annual Reviews in Control 2021.